Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

Preparação da carga dataframe

In [ ]:
# Lista com o dicionario de dados 
colunas = [
    "ID_CLIENTE","TIPO_FUNCIONARIO","DIA_PAGAMENTO",
    "TIPO_ENVIO_APLICACAO","QUANT_CARTOES_ADICIONAIS",
    "TIPO_ENDERECO_POSTAL","SEXO","ESTADO_CIVIL",
    "QUANT_DEPENDENTES","NIVEL_EDUCACIONAL",
    "ESTADO_NASCIMENTO","CIDADE_NASCIMENTO",
    "NACIONALIDADE","ESTADO_RESIDENCIAL",
    "CIDADE_RESIDENCIAL","BAIRRO_RESIDENCIAL",
    "FLAG_TELEFONE_RESIDENCIAL",
    "CODIGO_AREA_TELEFONE_RESIDENCIAL",
    "TIPO_RESIDENCIA","MESES_RESIDENCIA",
    "FLAG_TELEFONE_MOVEL","FLAG_EMAIL",
    "RENDA_PESSOAL_MENSAL","OUTRAS_RENDAS",
    "FLAG_VISA","FLAG_MASTERCARD","FLAG_DINERS",
    "FLAG_AMERICAN_EXPRESS","FLAG_OUTROS_CARTOES",
    "QUANT_CONTAS_BANCARIAS",
    "QUANT_CONTAS_BANCARIAS_ESPECIAIS",
    "VALOR_PATRIMONIO_PESSOAL","QUANT_CARROS",
    "EMPRESA","ESTADO_PROFISSIONAL",
    "CIDADE_PROFISSIONAL","BAIRRO_PROFISSIONAL",
    "FLAG_TELEFONE_PROFISSIONAL",
    "CODIGO_AREA_TELEFONE_PROFISSIONAL",
    "MESES_NO_TRABALHO","CODIGO_PROFISSAO",
    "TIPO_OCUPACAO","CODIGO_PROFISSAO_CONJUGE",
    "NIVEL_EDUCACIONAL_CONJUGE",
    "FLAG_DOCUMENTO_RESIDENCIAL","FLAG_RG",
    "FLAG_CPF","FLAG_COMPROVANTE_RENDA",
    "PRODUTO","FLAG_REGISTRO_ACSP",
    "IDADE","CEP_RESIDENCIAL_3",
    "CEP_PROFISSIONAL_3",
    "ROTULO_ALVO_MAU"
]

df = pd.read_csv(
    "saida_100_linhas.csv",
    header=0,
    names=colunas,
    na_values=[" ", ""]
)




df.head()

Visualização Inicial

In [ ]:
print("Linhas:", df.shape[0])
print("Colunas:", df.shape[1])



df.info()

Valores Ausentes

In [ ]:
missing = (
    df.isnull()
      .mean()
      .sort_values(ascending=False)
      .to_frame("Percentual")
)

missing.head(20)

Convert colunas no data frame para numerico

In [ ]:
for coluna in df.columns:
    try:
        df[coluna] = pd.to_numeric(df[coluna])
    except:
        pass

Estatistica Descritiva

In [ ]:
df.describe().T

Analise da distribuição da variavel alvo

In [ ]:
df["ROTULO_ALVO_MAU"].value_counts()

In [ ]:
df["ROTULO_ALVO_MAU"].value_counts()

df["ROTULO_ALVO_MAU"].value_counts().plot(
    kind="bar",
    figsize=(6,4),
    title="Distribuição da Classe"
)

plt.show()

Proporção de clientes bom pagadores maior qwe de maus pagadores

Análise da Renda por classe

In [ ]:
df.groupby("ROTULO_ALVO_MAU")[
    "RENDA_PESSOAL_MENSAL"
].agg([
    "count",
    "mean",
    "median",
    "min",
    "max"
])

In [ ]:
sns.boxplot(
    data=df,
    x="ROTULO_ALVO_MAU",
    y="RENDA_PESSOAL_MENSAL"
)

plt.title("Renda por Classe")
plt.show()

Conclusão: A variável RENDA_PESSOAL_MENSAL está muito concentrada em valores baixos para as duas classes (ROTULO_ALVO_MAU = 0 e ROTULO_ALVO_MAU = 1), mas existem muitos outliers altos que distorcem a escala do gráfico. 
A renda mensal, isoladamente, não parece ser uma variável suficiente para distinguir claramente clientes bons de clientes maus.
Uma possibilidade para gerar um gráfico mais adequado a como a variavel esta (sem recorrer a nenhuma tecnica de normalização dos dados) e adotar a escala logaritmica ou restringir usando um limite no eixo Y

In [ ]:
sns.boxplot(
    data=df,
    x="ROTULO_ALVO_MAU",
    y="RENDA_PESSOAL_MENSAL"
)

plt.ylim(0, df["RENDA_PESSOAL_MENSAL"].quantile(0.95))
plt.title("Renda por Classe sem outliers extremos")
plt.show()

Análise da Idade do cliente

In [ ]:
df.groupby("ROTULO_ALVO_MAU")[
    "IDADE"
].agg([
    "count",
    "mean",
    "median",
    "min",
    "max"
])

In [ ]:
sns.boxplot(
    data=df,
    x="ROTULO_ALVO_MAU",
    y="IDADE"
)

plt.title("Idade por Classe")
plt.show()

Análise e Tratamento varieis categoricas

In [ ]:
print( df["ESTADO_CIVIL"].unique())
print( df["EMPRESA"].unique())
print( df["PRODUTO"].unique())
print( df["SEXO"].unique())
print( df["TIPO_ENVIO_APLICACAO"].unique())




In [ ]:
#ajustes nas colunas sexo e tipo envio aplicação
df.loc[df["TIPO_ENVIO_APLICACAO"] == 'Web', "TIPO_ENVIO_APLICACAO"] = '1'
df.loc[df["TIPO_ENVIO_APLICACAO"] == 'Carga', "TIPO_ENVIO_APLICACAO"] = '2'



In [ ]:
df.loc[df["SEXO"] == 'M', "SEXO"] = '1'
df.loc[df["SEXO"] == 'F', "SEXO"] = '0'
df.loc[df["SEXO"] == 'N', "SEXO"] = '-1'
df.loc[df["SEXO"] == ' ', "SEXO"] = '-1'
# -1 SEXO NÃO INFORMADO

In [ ]:
variaveis_categoricas = [
    "ESTADO_CIVIL",
    "EMPRESA",
    "PRODUTO",
    "TIPO_ENVIO_APLICACAO",
    "SEXO"
]

for var in variaveis_categoricas:

    tabela = (
        df.groupby(var)["ROTULO_ALVO_MAU"]
          .agg(
              quantidade="count",
              maus="sum",
              taxa_mau="mean"
          )
          .sort_values(
              "taxa_mau",
              ascending=False
          )
    )

    print(f"\n===== {var} =====")
    display(tabela)

Calculo da correlação variavel alvo (ROTULO_ALVO_MAU)

In [ ]:
numericas = df.select_dtypes(include=np.number)

correlacoes = (
    numericas.corr(method="spearman")
              ["ROTULO_ALVO_MAU"]
              .sort_values(
                  key=lambda x: abs(x),
                  ascending=False
              )
)

print(correlacoes)

In [ ]:
correlacoes.head(20).sort_values().plot(
    kind="barh",
    figsize=(10,8)
)

plt.title(
    "Correlação com ROTULO_ALVO_MAU"
)

plt.show()

In [ ]:
# Visualizando a matriz em um mapa de calor (heatmap)

data = correlacoes.head(20).sort_values().to_frame()
plt.figure(figsize=(25, 20))
sns.heatmap(data, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlação')
plt.show()
